# 🚗 Truck Blind Spot Detection - Notebook Demo

Notebook này hướng dẫn cách sử dụng `BlindSpotPipeline` để phát hiện vật thể trong vùng điểm mù trực tiếp từ code Python.

--- 
### 🎁 Cách tận dụng Google Colab Pro (H100/A100)
1. Truy cập **Runtime** -> **Change runtime type** -> Chọn GPU cao nhất (Vd: **H100** hoặc **A100**).
2. Chạy cell **Setup for Google Colab** để cài đặt môi trường.
3. Code bên dưới sẽ tự động phát hiện GPU và khởi tạo model YOLOv9 trên GPU để đạt FPS tối đa (~30-50 FPS).

### 0. Setup for Google Colab (Chỉ chạy nếu dùng Colab)

In [ ]:
import os
import sys

# 1. Mount Google Drive (Nếu bạn lưu code trên Drive)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Thay đổi %cd tới thư mục project của bạn
    # %cd /content/drive/MyDrive/truck_blind_spot
except ImportError:
    print("Not on Colab. Skipping Drive mount.")

# 2. Cài đặt dependencies (Nhanh hơn trên Colab Pro)
if os.path.exists('requirements.txt'):
    !pip install -r requirements.txt --quiet
    print("✅ Môi trường đã sẵn sàng!")
else:
    print("⚠️ Không tìm thấy file requirements.txt, hãy kiểm tra lại thư mục hiện tại.")

### 1. Import dependencies & Check GPU

In [ ]:
import cv2
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from src.pipeline import BlindSpotPipeline
from IPython.display import display, Image, clear_output

%matplotlib inline

# 🚀 Tận dụng Colab Pro GPU
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"📌 Device: {device}")
if device == "cuda:0":
    print(f"🔥 GPU Model: {torch.cuda.get_device_name(0)}")

### 2. Khởi tạo Pipeline

In [ ]:
pipeline = BlindSpotPipeline(
    weights_path="weights/best_small.pt",
    roi_config_path="configs/roi.json",
    classes_config_path="configs/classes.yaml",
    device=device, # Tận dụng GPU
    conf_threshold=0.25,
    iou_threshold=0.45
)
print("✅ Pipeline initialized on GPU!")

### 3. Chạy video demo (Hiển thị trong Notebook)

In [ ]:
video_path = "assets/videos/demo.mp4"
cap = cv2.VideoCapture(video_path)

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
            
        annotated_frame, _ = pipeline.process_frame(frame)
        
        # Mã hóa thành jpeg để hiển thị nhanh
        _, buffer = cv2.imencode('.jpg', annotated_frame)
        display(Image(data=buffer))
        
        clear_output(wait=True)
finally:
    cap.release()
    print("🎬 Video processed.")